# 🇸🇩 استوديو استنساخ وتوليد الصوت السوداني الذكي (XTTS-v2)
### مُهندس الصوت السوداني الأصيل — استنساخ النبرة والمخارج التلقائية من مكتبة التسجيلات المرجعية
---
**طريقة الاستخدام السريعة (3 خطوات فقط):**
1. اضغط زر التشغيل `▶` على **الخلية 1** (تثبيت المحرك والربط).
2. اضغط زر التشغيل `▶` على **الخلية 2** (تحميل النموذج العصبي XTTS-v2 على T4 GPU).
3. اضغط زر التشغيل `▶` على **الخلية 3** (الواجهة التفاعلية لتوليد وسماع الصوت فوراً).

## 1️⃣ الخلية الأولى: تثبيت محرك الصوت الحديث وربط Google Drive
*(تستغرق حوالي دقيقة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [3]:
import os
# تفعيل الموافقة التلقائية على ترخيص النموذج العصبي
os.environ['COQUI_TOS_AGREED'] = '1'

# تثبيت حزمة coqui-tts الحديثة المتوافقة 100% مع بايثون الجديد في Colab
!pip install -q coqui-tts soundfile gradio

# ربط Google Drive للوصول للعينات المرجعية والمعجم السوداني
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

print('\n✅ تم تثبيت محرك الصوت coqui-tts وربط Google Drive بنجاح!')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ تم تثبيت محرك الصوت coqui-tts وربط Google Drive بنجاح!


## 2️⃣ الخلية الثانية: تحميل نموذج XTTS-v2 العصبي على كرت الشاشة (GPU)
*(تستغرق دقيقة واحدة — انتظر ظهور علامة الصح الخضراء `✓`)*

In [4]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

import torch
from TTS.api import TTS
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 المحرك يعمل على: {device}')
if device == 'cuda':
    print(f'⚡ كرت الشاشة المكتشف: {torch.cuda.get_device_name(0)}')

print('⏳ جاري تحميل نموذج Coqui XTTS-v2 العصبي...')
xtts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

# مسار العينات المرجعية والمعجم السوداني المستخرج من الكتب
REF_DIR = '/content/drive/MyDrive/Sudanese_Voice_Project/corpus/references'
KB_PATH = '/content/drive/MyDrive/Sudanese_Voice_Project/sudanese_master_knowledge_base.json'

sudanese_kb = {}
if os.path.exists(KB_PATH):
    with open(KB_PATH, 'r', encoding='utf-8') as f:
        sudanese_kb = json.load(f)
    print(f'📚 تم تحميل المعجم اللغوي السوداني ({sudanese_kb.get("metadata", {}).get("total_extracted_words", 0):,} كلمة)')

print('\n✅ النموذج جاهز للعمل والإنتاج الصوتي الفوري!')

🚀 المحرك يعمل على: cuda
⚡ كرت الشاشة المكتشف: Tesla T4
⏳ جاري تحميل نموذج Coqui XTTS-v2 العصبي...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


📚 تم تحميل المعجم اللغوي السوداني (304,297 كلمة)

✅ النموذج جاهز للعمل والإنتاج الصوتي الفوري!


## 3️⃣ الخلية الثالثة: الواجهة التفاعلية الفورية (Gradio Web UI) والتجربة المباشرة
*(شغل هذه الخلية وستفتح لك واجهة كاملة بالمتصفح لكتابة وسماع أي نص سوداني فوراً)*

In [ ]:
import gradio as gr
import os

# عينات نقية مركزة (8 ثوانٍ) تم استخلاصها بدقة لأعلى جودة في مخارج الحروف
PRESETS = {
    '🎙️ بودكاست وإعلام رصين (نقي 8 ثوانٍ)': os.path.join(REF_DIR, 'clean_podcast_8s.wav'),
    '🎭 دراما ومحادثة حية (نقي 8 ثوانٍ)': os.path.join(REF_DIR, 'clean_drama_8s.wav'),
    '☕ وناسة وجلسة شبابية (نقي 8 ثوانٍ)': os.path.join(REF_DIR, 'clean_wanasa_8s.wav'),
    '😄 فكاهة وشخصيات طريفة (شيلابان)': os.path.join(REF_DIR, 'clean_comedy_8s.wav')
}

def generate_sudanese_voice(text, speaker_choice, speed, clarity, custom_audio):
    ref_wav = custom_audio if (custom_audio and os.path.exists(custom_audio)) else PRESETS.get(speaker_choice, list(PRESETS.values())[0])
    
    if not os.path.exists(ref_wav):
        ref_wav = os.path.join(REF_DIR, 'podcast_alsudanpodcast.wav')
        
    out_path = '/content/drive/MyDrive/Sudanese_Voice_Project/cloned_sudanese.wav'
    
    print(f'🎙️ المتحدث المرجعي: {os.path.basename(ref_wav)}')
    print(f'📝 النص الأصلي بدون تشويه: {text}')
    
    # توليد الصوت مع ضبط دقة المخارج (temperature=0.6 يمنع تلعثم الحروف)
    xtts.tts_to_file(
        text=text,
        speaker_wav=ref_wav,
        language='ar',
        file_path=out_path,
        speed=speed,
        temperature=clarity,
        repetition_penalty=2.0
    )
    return out_path

demo = gr.Interface(
    fn=generate_sudanese_voice,
    inputs=[
        gr.Textbox(
            label='✍️ اكتب النص باللهجة السودانية (اكتب بالحروف العربية الطبيعية):',
            value='أهلاً بيكم يا جماعة، دحين الموضوع دا سمح بالحيل، وقاعدين هنا عشان نتونس فيهو برواقة ونفهم الحاصل شنو.',
            lines=3
        ),
        gr.Dropdown(
            choices=list(PRESETS.keys()),
            value=list(PRESETS.keys())[0],
            label='🗣️ اختر المتحدث السوداني (عينات نقية مركزة 8 ثوانٍ):'
        ),
        gr.Slider(minimum=0.7, maximum=1.3, value=1.0, step=0.05, label='⚡ سرعة الإلقاء (Speed)'),
        gr.Slider(minimum=0.3, maximum=0.9, value=0.6, step=0.05, label='🎯 وضوح مخارج الحروف (0.6 يعطي أعلى صفاء ومخارج سليمة)'),
        gr.Audio(sources=['upload'], type='filepath', label='🎙️ أو ارفع تسجيلك الصوتي الخاص (6-10 ثوانٍ نقية) لاستنساخ حنجرتك بدقة:')
    ],
    outputs=gr.Audio(label='🔊 استمع للصوت السوداني الجديد عالي النقاء والوضوح:', type='filepath'),
    title='🇸🇩 استوديو الصوت والذكاء الاصطناعي السوداني - الإصدار النقي',
    description='توليد نطق سوداني واقعي عالي النقاء بمعايرة مخارج الحروف عبر عينات مرجعية مركزة (8 ثوانٍ)'
)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://07ff0e32a5132e1979.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
🎙️ توليد بصوت: podcast_alsudanpodcast.wav
📝 النص: أهلاً بيكم يا جماعة، دحين الموضوع دا سَمِحْ بالحيل، وگاعدين هنا عشان نتونس فيهو برواقة ونفهم الحاصل شنو.
🎙️ توليد بصوت: comedy_shailaban.wav
📝 النص: أهلاً بيكم يا جماعة، دحين الموضوع دا سَمِحْ بالحيل، وگاعدين هنا عشان نتونس فيهو برواقة ونفهم الحاصل شنو.
